# Information Geometry (Fisher Metric)

**Concept 40 of the math-foundations learning map.**

We treat a parameterized family of distributions $\{p_\theta\}$ as a Riemannian manifold. The **Fisher information matrix**
$$
F(\theta) = \mathbb{E}_{p_\theta}\!\left[(\nabla_\theta \log p_\theta)(\nabla_\theta \log p_\theta)^\top\right] = -\mathbb{E}_{p_\theta}\!\left[\nabla^2 \log p_\theta\right]
$$
is the metric tensor. The natural gradient is $\widetilde{\nabla} L = F^{-1} \nabla L$.

In this notebook we work with the simplest non-trivial family $\mathcal{N}(\mu, 1)$, parameterized by the mean $\mu \in \mathbb{R}$, and verify two facts:
1. $F(\mu) = 1$ analytically and numerically.
2. $D_{\mathrm{KL}}(\mathcal{N}(0,1) \,\|\, \mathcal{N}(\epsilon,1)) = \tfrac{1}{2}\epsilon^2$, matching the Fisher quadratic form.


## 1. Setup (stdlib only)
We use `math` and `random` — no NumPy.


In [ ]:
import math
import random

def log_p(x, mu):
    return -0.5 * math.log(2.0 * math.pi) - 0.5 * (x - mu) ** 2

# Sanity check: density at mode
print("log p_0(0) =", log_p(0.0, 0.0))
print("p_0(0)     =", math.exp(log_p(0.0, 0.0)))


## 2. Fisher information for $\mathcal{N}(\mu, 1)$

Analytical derivation:
- $\partial_\mu \log p_\mu(x) = (x - \mu)$
- $\partial_\mu^2 \log p_\mu(x) = -1$
- $F(\mu) = -\mathbb{E}[\partial_\mu^2 \log p_\mu] = 1$.

Numerical: estimate $F(\mu) = \mathbb{E}_{x \sim p_\mu}[-\partial_\mu^2 \log p_\mu(x)]$ via Monte Carlo with finite-difference Hessian.


In [ ]:
def numerical_second_derivative(f, mu, h=1e-3):
    return (f(mu + h) - 2.0 * f(mu) + f(mu - h)) / (h * h)

def fisher_numerical(mu, n_samples=20000, seed=0):
    rng = random.Random(seed)
    total = 0.0
    for _ in range(n_samples):
        x = rng.gauss(mu, 1.0)
        # f(mu) = -log p_mu(x); its second derivative is -d^2 log p / dmu^2.
        f = lambda m, x=x: -log_p(x, m)
        total += numerical_second_derivative(f, mu)
    return total / n_samples

mu = 0.0
F_hat = fisher_numerical(mu)
print(f"Analytical F(mu) = 1.0")
print(f"Numerical  F(mu) = {F_hat:.4f}")
print(f"|error|          = {abs(F_hat - 1.0):.4f}")


## 3. Local KL = $\tfrac{1}{2}\epsilon^\top F(\theta)\, \epsilon$

For unit-variance Gaussians:
$$D_{\mathrm{KL}}(\mathcal{N}(\mu_p, 1) \,\|\, \mathcal{N}(\mu_q, 1)) = \tfrac{1}{2}(\mu_p - \mu_q)^2.$$

This matches the Fisher prediction $\tfrac{1}{2}\epsilon^2 \cdot F = \tfrac{1}{2}\epsilon^2$ exactly (the family is quadratic in $\mu$, so there are no higher-order terms).


In [ ]:
def kl_gaussian_unit_var(mu_p, mu_q):
    return 0.5 * (mu_p - mu_q) ** 2

print(f"{'eps':>10}  {'D_KL':>14}  {'eps^2/2':>14}  {'ratio':>10}")
for eps in (1e-1, 1e-2, 1e-3, 1e-4):
    kl = kl_gaussian_unit_var(0.0, eps)
    approx = 0.5 * eps * eps
    ratio = kl / approx
    print(f"{eps:10.0e}  {kl:14.6e}  {approx:14.6e}  {ratio:10.6f}")


## 4. Natural gradient on a tiny example

Suppose we want to fit $\mu$ by minimizing $L(\mu) = \tfrac{1}{2}(\mu - \mu^*)^2$ with $\mu^* = 2$.

- Ordinary gradient: $\nabla L = (\mu - \mu^*)$.
- Natural gradient: $\widetilde{\nabla} L = F(\mu)^{-1} \nabla L = (\mu - \mu^*)$.

For $\mathcal{N}(\mu, 1)$ they agree because $F = 1$. Try changing the family (e.g. add variance) to see them diverge.


In [ ]:
mu_star = 2.0
mu = 0.0
eta = 0.5
for step in range(6):
    grad = mu - mu_star
    nat_grad = grad / 1.0  # F = 1
    mu = mu - eta * nat_grad
    print(f"step {step+1}: mu = {mu:.4f}, loss = {0.5*(mu-mu_star)**2:.4f}")


## 5. Takeaways

- Fisher information $F(\theta)$ is the natural Riemannian metric on a statistical manifold.
- Locally, KL divergence equals the Fisher quadratic form (Theorem in `lesson.tex`).
- Natural-gradient descent $\theta \leftarrow \theta - \eta F^{-1} \nabla L$ is invariant under reparameterization and is the foundation of methods like TRPO and K-FAC.
